In [1]:
from sklearn.model_selection import train_test_split
from shopsense.models.churn_model import build_churn_preprocessing_pipeline, train_churn_model, evaluate_churn_model
from shopsense.models.segmentation_model import prepare_clustering_features, find_optimal_k, train_segmentation_model, profile_clusters

# Assuming master_df is loaded/generated from previous steps. For standalone execution:
from shopsense.data_generator import generate_customers, generate_transactions, generate_events
from shopsense.features import compute_rfm_features, compute_behavioral_features, compute_transaction_features, build_master_feature_table
df_c = generate_customers(2000, random_state=42)
df_t = generate_transactions(df_c, random_state=42)
df_e = generate_events(df_c, random_state=42)
master_df = build_master_feature_table(df_c, compute_rfm_features(df_t, '2023-12-31'), compute_behavioral_features(df_e, df_t, '2023-12-31'), compute_transaction_features(df_t, '2023-12-31'))

# 1. Churn Modeling
target = 'churn_label'
categorical_features = ['gender', 'city', 'acquisition_channel', 'rfm_segment', 'preferred_device', 'preferred_category', 'preferred_payment']
numeric_features = [col for col in master_df.columns if col not in categorical_features + [target]]

X = master_df.drop(columns=[target])
y = master_df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = build_churn_preprocessing_pipeline(numeric_features, categorical_features)
churn_model = train_churn_model(X_train, y_train, preprocessor)
print("--- Churn Model Evaluation ---")
display(evaluate_churn_model(churn_model, X_test, y_test))

# 2. Segmentation
X_scaled, feature_names, _ = prepare_clustering_features(master_df)
kmeans, labels = train_segmentation_model(X_scaled, n_clusters=2)
print("\n--- Customer Segments ---")
display(profile_clusters(master_df, labels, feature_names)[['cluster_size', 'churn_rate', 'monetary_total']])

c:\Users\surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/05 22:45:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/05 22:45:12 INFO mlflow.store.db.utils: Updating database tables
2026/06/05 22:45:17 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8bcd5a0a570044cc901b0ce526c7899d', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/05 22:45:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python canno

--- Churn Model Evaluation ---


2026/06/05 22:45:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\surya\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


{'roc_auc': 1.0,
 'pr_auc': 1.0,
 'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'balanced_accuracy': 1.0,
 'confusion_matrix': [[300, 0], [0, 100]],
 'classification_report': '              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00       300\n           1       1.00      1.00      1.00       100\n\n    accuracy                           1.00       400\n   macro avg       1.00      1.00      1.00       400\nweighted avg       1.00      1.00      1.00       400\n'}

2026/06/05 22:45:21 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'fe0ee448ffc04a3798a20363dd543be3', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/06/05 22:45:26 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.



--- Customer Segments ---


,cluster_size,churn_rate,monetary_total
0,506,0.988142,5202.926569
1,1494,0.000000,24602.226811
